# Overfitting, Underfitting and the Bias-Variance Trade-off
Why a model can be too simple or too complex, and how to tell. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A household records the outdoor temperature and its electricity use each day. Cold days need heating, hot days need cooling, so the pattern is a curve.

### 300 days of electricity use
- `electricity.csv`: 300 days: temperature in degrees Celsius, electricity used in kWh
- `df.sample(30, random_state=4)`: just 30 random days, to start with: small data makes overfitting easy to see

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "overfitting-and-bias-variance/electricity.csv")
df = pd.read_csv(url)
small = df.sample(30, random_state=4)
print(df.shape, small.shape)
small.head()

### Look at the data
- `the shape`: high at both ends, lowest around 18 degrees: a curve, not a straight line

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(small["temperature"], small["electricity_kwh"])
plt.xlabel("temperature (C)")
plt.ylabel("electricity (kWh)")
plt.title("Heating when cold, cooling when hot")
plt.show()

## 2. Three models: too simple, just right, too complex
**Polynomial features** add temperature squared, cubed and so on as extra inputs, so linear regression can draw curves. The **degree** is the highest power used.

### Split the 30 days
- `test_size=0.3`: hide 30% of the days to test on

In [ ]:
from sklearn.model_selection import train_test_split

X = small[["temperature"]]
y = small["electricity_kwh"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)
print(len(X_train), "training days,", len(X_test), "test days")

### Fit degree 1, 2 and 15
- `PolynomialFeatures(degree)`: adds temperature squared, cubed... up to the degree
- `StandardScaler()`: puts those very different-sized columns on the same scale
- `mean_squared_error`: the average squared miss: lower is better
- `degree 1`: a straight line: bad on training AND test days (underfitting)
- `degree 2`: a simple curve: good on both
- `degree 15`: tiny training error, huge test error: it memorised the training days (overfitting)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree), StandardScaler(),
                         LinearRegression())

for d in [1, 2, 15]:
    m = poly_model(d).fit(X_train, y_train)
    tr = mean_squared_error(y_train, m.predict(X_train))
    te = mean_squared_error(y_test, m.predict(X_test))
    print(f"degree {d:>2}:  train error {tr:7.1f}   test error {te:9.1f}")

### See the three models
- `np.linspace(0, 35, 300)`: 300 evenly spaced temperatures, to draw smooth curves
- `marker="x"`: test days drawn as crosses
- `the picture`: degree 15 wiggles through every training dot and misses the crosses

In [ ]:
import numpy as np

grid = pd.DataFrame({"temperature": np.linspace(0, 35, 300)})
plt.scatter(X_train, y_train, label="training days")
plt.scatter(X_test, y_test, marker="x", s=80, label="test days")
for d in [1, 2, 15]:
    m = poly_model(d).fit(X_train, y_train)
    plt.plot(grid, m.predict(grid), label=f"degree {d}")
plt.ylim(0, 40); plt.legend()
plt.xlabel("temperature (C)"); plt.ylabel("electricity (kWh)")
plt.show()

## 3. Bias and variance
**Bias**: error from a model that's too simple, off in the same way whatever data it sees. **Variance**: how much the model changes when the training data changes.

### Train on five different samples
- `plt.subplots(1, 2)`: two charts side by side
- `df.sample(20, random_state=seed)`: five different random sets of 20 days
- `degree 1`: the five lines agree with each other, but all miss the curve: high bias, low variance
- `degree 15`: the five curves are wildly different: low bias, high variance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, d in zip(axes, [1, 15]):
    for seed in range(5):
        s = df.sample(20, random_state=seed)
        m = poly_model(d).fit(s[["temperature"]], s["electricity_kwh"])
        ax.plot(grid, m.predict(grid))
    ax.set_ylim(0, 40); ax.set_title(f"degree {d}: five samples")
plt.show()

## 4. Finding the sweet spot
Try every degree and compare training and test error.

### Error against complexity
- `range(1, 13)`: degrees 1 to 12
- `plt.yscale("log")`: a log scale, because the test error grows enormous
- `training error`: keeps falling as the model gets more complex
- `test error`: falls, then rises: the lowest point is the sweet spot

In [ ]:
train_err, test_err, degrees = [], [], range(1, 13)
for d in degrees:
    m = poly_model(d).fit(X_train, y_train)
    train_err.append(mean_squared_error(y_train, m.predict(X_train)))
    test_err.append(mean_squared_error(y_test, m.predict(X_test)))
plt.plot(degrees, train_err, marker="o", label="training error")
plt.plot(degrees, test_err, marker="o", label="test error")
plt.yscale("log"); plt.legend()
plt.xlabel("degree (model complexity)"); plt.ylabel("error")
plt.show()

## 5. Fixing overfitting
More data, a simpler model, or regularisation (penalising big weights). Here: more data.

### Degree 15 with 21 days vs 300 days
- `X_train.join(y_train)`: our 21 training days, as one table
- `df`: all 300 days
- `the picture`: with lots of data, even the complex model is forced into a sensible curve

In [ ]:
for data, name in [(X_train.join(y_train), "21 days"), (df, "300 days")]:
    m = poly_model(15).fit(data[["temperature"]], data["electricity_kwh"])
    plt.plot(grid, m.predict(grid), label=f"degree 15, {name}")
plt.scatter(df["temperature"], df["electricity_kwh"], s=8, color="gray")
plt.ylim(0, 40); plt.legend(); plt.show()

## Try it yourself
1. Try degree 4 and degree 8. Where do they sit between 'just right' and 'overfitting'?
2. In the 'more data' cell, use 100 days instead of 300. How wiggly is degree 15 now?
3. Change random_state in the split. Does degree 2 still win?